# IPTA DR2 — joint full-basis decentering validation (J1640+2224)

New-API rebuild of the decentering validation, on the real EPTA-DR2 J1640+2224
data. It exercises the timing-coordinate-charts / geometry feature end to end and
reproduces — then **fixes** — the geometry pathology that broke the earlier
decentering run:

- typed inference plan + one coordinate chart per axis (`chart_summary`);
- the transport-center report read **chart-type-first** (a large `center_z` is a
  boundary diagnostic only for `prior_pit` axes; `affine_normal` has no boundary);
- **the headline**: `F0`/`F1` on wide uniform charts give a catastrophic off-mode
  geometry (the ≈2.5-width axis of the Stage 0 forensic); `identically_linear=`
  turns it into a far tamer target;
- fixed-expansion refinement at the WN+hyper MPE;
- geometry certification over box hyper-probes + a standalone JSON/NPZ report;
- block-dense-mass NUTS with chain-preserving diagnostics (each chain plotted
  separately);
- pivoted vs 1/yr red-noise amplitude.

Requires the EPTA-DR2 J1640 par/tim and the devcontainer stack. Sampling cells
are modest; scale `num_warmup`/`num_samples`/`num_chains` up for science.


In [ ]:
import os
os.environ.setdefault("JAX_ENABLE_X64", "1")
from pathlib import Path

import jax
import numpy as np
import matplotlib.pyplot as plt

import discovery as ds
from discovery import transport as dst
from metapulsar import create_metapulsar
from metapulsar.sandbox_tempo2 import configure_logging
from nltiming import (
    NonLinearTimingModel, TimingInference, box_hyper_probe_points, certify_joint_geometry, transport_center_report,
    write_geometry_report, read_geometry_report, refine_timing_expansion,
)
from nltiming.sampling import numpyro as N

ds.config(kernels="metamath")
configure_logging(level="WARNING")

# Real IPTA-DR2 J1640+2224 (EPTA v2.2). Find the repo root from the CWD.
_here = Path.cwd()
_repo = next(p for p in (_here, *_here.parents) if (p / "data" / "ipta-dr2").is_dir())
PAR = _repo / "data/ipta-dr2/EPTA_v2.2/J1640+2224/J1640+2224.par"
TIM = _repo / "data/ipta-dr2/EPTA_v2.2/J1640+2224/J1640+2224_all.tim"
assert PAR.exists(), f"J1640 par not found: {PAR}"

mp = create_metapulsar(
    {"epta": [{"par": str(PAR), "tim": str(TIM), "timing_package": "tempo2"}]},
    use_pulse_numbers="no",
)
nd = {f"{mp.name}_efac": 1.0, f"{mp.name}_log10_t2equad": -8.0}
reference = dst.reference_noise_frozen(
    ds.makenoise_measurement_simple(mp, nd), nd, description="J1640 WN reference")
center = {f"{mp.name}_rednoise_log10_A": -14.0, f"{mp.name}_rednoise_gamma": 3.5}
print(f"{mp.name}: {len(mp.toas)} TOAs, {len(mp.fitpars)} fitpars")
print("fitpars:", list(mp.fitpars))


## 1. Joint full-basis context and the coordinate charts

`sample_all()` samples every timing axis (the joint full-basis / decentering
model). Each proper axis carries one chart; note which axes are `prior_pit`
(bounded/uniform cheat prior, a local map) vs `affine_normal` (Gaussian delta
prior, globally affine). `identically_linear` lists the registry/engine-certified
linear axes.


In [ ]:
ctx = NonLinearTimingModel(
    engines="jug", inference=TimingInference.sample_all(), name="timing"
).for_pulsar(mp)

print(f"{'axis':12s} {'disposition':12s} {'chart':14s} {'lin?':5s} prior")
for d in ctx.chart_summary():
    print(f"{d['name']:12s} {d['disposition']:12s} {d['chart']:14s} "
          f"{str(d['identically_linear']):5s} {d['prior_family']}")
print("\nidentically linear:", ctx.identically_linear)
print("resolved tempo2_native:", ctx.model.resolved_tempo2_native)


## 2. Transport-center report — chart type first

For each axis: is the conditioned center inside its chart? Report the **chart
type first**. An `affine_normal` axis is interior for any finite center (a
Gaussian mean shift). Only a `prior_pit` axis has a finite boundary, and only
there is a large `|center_z|` a saturation warning. (We fix the noise for a fast
build; the red-noise hyper are free below.)


In [ ]:
psl = ds.PulsarLikelihood([
    mp.residuals,
    ds.makenoise_measurement_simple(mp, nd),
    ds.makegp_fourier(mp, ds.powerlaw, 10, name="rednoise"),
    *ctx.discovery_signals(joint=True),
])
jm = N.joint_model(psl, ctx, reference_noise=reference, fixed=nd)

for a in transport_center_report(ctx, jm.transport, center):
    kind = "affine (no boundary)" if a.chart == "affine_normal" else \
        ("PIT interior" if a.interior else "PIT SATURATED")
    print(f"{a.name:12s} {a.chart:14s} center_z={a.center_z:+7.3f}  {kind}")


## 3. The headline: `identically_linear` fixes the off-mode geometry

`F0`/`F1` are exactly linear in phase but are **not** in the conservative
fallback registry, so by default they sit on wide uniform `prior_pit` charts. Off
the mode — where the certifier probes — that raises the joint target's curvature
(this is the ≈2.5-width axis that broke the earlier decentering run; on a poorly
constrained pulsar it blows up by orders of magnitude, cf.
`02_geometry_certification_and_pivot.ipynb`).

Declaring the linear axes identically linear (unioned with the auto-derived set,
since `identically_linear=` is authoritative) flips them to `affine_normal`. Watch
the Hessian eigenvalues collapse toward 1 and the remainder RMS drop ~100×. On
real J1640 the report still does **not** fully pass: the *binary* parameters
(`PB`, `A1`, …) remain genuinely nonlinear on `prior_pit` charts, and the WN-only
reference cannot precondition the timing↔red-noise cross-curvature. That is the
honest, actionable residual — not something to force under a bar.


In [ ]:
bounds = {f"{mp.name}_rednoise_log10_A": (-18.0, -11.0),
          f"{mp.name}_rednoise_gamma": (0.0, 7.0)}
probes = box_hyper_probe_points(center, bounds)[:1]
report_default = certify_joint_geometry(jm, ctx, hyper_points=probes)

linear_axes = {"F0", "F1", "RAJ", "DECJ", "PMRA", "PMDEC", "PX"} & set(mp.fitpars)
declared = sorted(set(ctx.identically_linear) | linear_axes)
ctx_lin = NonLinearTimingModel(
    engines="jug", inference=TimingInference.sample_all(),
    identically_linear=declared, name="timing",
).for_pulsar(mp)
psl_lin = ds.PulsarLikelihood([
    mp.residuals, ds.makenoise_measurement_simple(mp, nd),
    ds.makegp_fourier(mp, ds.powerlaw, 10, name="rednoise"),
    *ctx_lin.discovery_signals(joint=True)])
jm_lin = N.joint_model(psl_lin, ctx_lin, reference_noise=reference, fixed=nd)
report_lin = certify_joint_geometry(jm_lin, ctx_lin, hyper_points=probes)


def _row(lbl, r):
    return (f"  {lbl:22s} rms={r.max_residual_remainder_rms:11.4g}"
            f"  H_eig=[{r.xi_hessian_eigen_min:9.3g}, {r.xi_hessian_eigen_max:10.4g}]"
            f"  xi_grad={r.max_xi_gradient_inf_norm:10.4g}"
            f"  passed={r.passed}")


print("declared linear:", sorted(linear_axes))
print(_row("default (uniform)", report_default))
print(_row("identically_linear", report_lin))
print(f"  -> H eigenvalue max: {report_default.xi_hessian_eigen_max:.3g} "
      f"-> {report_lin.xi_hessian_eigen_max:.3g}")


## 4. Refine the fixed expansion at the WN+hyper MPE

The linearization expansion is refined against a **marginal** objective (red noise
integrated, timing via delay keys) at the fixed WN+hyper point. A well-fit par
file is already near the timing MPE. We continue with the identically-linear
context (`ctx_lin`).


In [ ]:
psl_marg = ds.PulsarLikelihood([
    mp.residuals, ds.makenoise_measurement_simple(mp, nd),
    ds.makegp_fourier(mp, ds.powerlaw, 10, name="rednoise"),
    *ctx_lin.discovery_signals(joint=False)])
objective = N.conditional_timing_potential(psl_marg, ctx_lin, fixed={**nd, **center})
refined = refine_timing_expansion(ctx_lin, negative_log_target_z=objective)
ctx_run = refined.context
z0 = np.asarray(ctx_lin.linearization.sampled_z_expansion)
z1 = np.asarray(ctx_run.linearization.sampled_z_expansion)
print("refine converged:", refined.converged, " |d z_e|:", float(np.linalg.norm(z1 - z0)))


## 5. Certify over box hyper-probes and write the standalone report

The cluster workflow writes a standalone JSON+NPZ geometry report beside its
other products; it round-trips and verifies its own digests, and is never part of
the run manifest or an `N.nuts` argument.


In [ ]:
psl_run = ds.PulsarLikelihood([
    mp.residuals, ds.makenoise_measurement_simple(mp, nd),
    ds.makegp_fourier(mp, ds.powerlaw, 10, name="rednoise"),
    *ctx_run.discovery_signals(joint=True)])
jm_run = N.joint_model(psl_run, ctx_run, reference_noise=reference, fixed=nd)

report = certify_joint_geometry(jm_run, ctx_run, hyper_points=box_hyper_probe_points(center, bounds)[:3])
print("passed:", report.passed, "| failures:", len(report.failures),
      "| rms:", round(report.max_residual_remainder_rms, 4))

outdir = Path(os.environ.get("TMPDIR", "/tmp")) / "j1640_decentering"
outdir.mkdir(parents=True, exist_ok=True)
jp, npzp = write_geometry_report(report, outdir / "geometry", overwrite=True)
print("wrote", jp.name, "+", npzp.name,
      "| roundtrip:", read_geometry_report(outdir / "geometry").model_fingerprint == report.model_fingerprint)


## 6. Block-dense-mass NUTS, chains preserved

`N.nuts` defaults to `dense_mass="auto"`: with ≥2 red-noise hyper it adapts a
dense block over `model.hyper_sites` only, leaving the intended-white `xi` on an
identity mass. We free the red-noise hyper here and keep chains separate — plot
each chain before any pooled corner (a frozen chain vs a moving chain is exactly
the §2.3 failure signature). Warmup/samples are small for the demo.


In [ ]:
rn_priors = {f"{mp.name}_rednoise_log10_A": (-18.0, -11.0),
             f"{mp.name}_rednoise_gamma": (0.0, 7.0)}
jm_free = N.joint_model(psl_run, ctx_run, reference_noise=reference,
                        priors=rn_priors, fixed=nd)
print("hyper sites (block mass):", jm_free.hyper_sites)

mcmc = N.nuts(jm_free, ctx_run, num_warmup=400, num_samples=400, num_chains=2,
              chain_method="sequential", progress_bar=True)
mcmc.run(jax.random.PRNGKey(0), extra_fields=N.NUTS_EXTRA_FIELDS)

diag = N.chain_diagnostics(mcmc, max_tree_depth=10)
print("tree-depth saturation:", round(diag['tree_depth_saturation_fraction'], 3),
      "| mean accept:", round(float(diag['accept_prob'].mean()), 3))

# Plot each chain's red-noise amplitude trace SEPARATELY (never pool first).
amp = f"{mp.name}_rednoise_log10_A"
sam = mcmc.get_samples(group_by_chain=True)[amp]
fig, axes = plt.subplots(1, sam.shape[0], figsize=(9, 3), sharey=True)
for i, ax in enumerate(np.atleast_1d(axes)):
    ax.plot(np.asarray(sam[i]), lw=0.6)
    ax.set_title(f"chain {i}")
    ax.set_xlabel("draw")
axes[0].set_ylabel("log10_A")
fig.tight_layout()


## 7. Pivoted vs 1/yr red-noise amplitude

Sampling `log10_A` at 1/yr correlates amplitude and slope. The sensitivity-
weighted pivot frequency decorrelates them via an affine unit-Jacobian map;
decode back to 1/yr for comparison.


In [ ]:
f, df, fmat = ds.fourierbasis(mp, 10)
w = ds.fourier_sensitivity_weights(fmat, dst.reference_noise(mp))
f_pivot = ds.sensitivity_weighted_pivot_frequency(np.asarray(f)[0::2], w)
print(f"pivot frequency: {f_pivot:.3e} Hz   (1/yr = {ds.const.fyr:.3e} Hz)")

param = ds.PowerLawParameterization(slope_pivot_frequency=f_pivot)
gamma = 3.5
for a_pivot in (-14.0, -14.5):
    a_ref = ds.reference_log10_amplitude(a_pivot, gamma, f_pivot=f_pivot, parameterization=param)
    print(f"  log10_A_pivot={a_pivot:.2f} (at f_pivot)  ==  log10_A={a_ref:.4f} (at 1/yr)")


## Summary

- **Read the chart first.** A large `|center_z|` is a boundary/saturation
  diagnostic *only* on `prior_pit` axes; `affine_normal` axes have no boundary,
  so a large mean shift there is neither "legitimate" nor "PIT saturation" — the
  earlier notebook's framing conflated the two.
- The joint full-basis geometry defect that broke the previous run was `F0`/`F1`
  on wide uniform charts. The fix is a **modeling decision** — declare the linear
  axes `identically_linear` — not a looser threshold or a deeper tree.
- On real J1640 that fix improves the geometry sharply but need not fully pass:
  the binary parameters are genuinely nonlinear and the WN-only reference cannot
  precondition the timing↔red-noise cross-curvature. The certifier names those as
  the next things to build (a red-noise-aware reference, or the multi-axis block
  bijector) — it does not hide them under a relaxed bar.
